# [V7] NFL Draft Prediction - Advanced Feature Engineering

Building on V6 (OOF AUC: 0.83096) with advanced features:
- **Polynomial features**: Squared terms for key metrics
- **Position-specific ratios**: Weight/Height, Bench/Weight, etc.
- **Performance consistency**: Std of z-scores across metrics
- **Elite athlete flags**: Top 10%/25% indicators by position
- **Multi-metric interactions**: 3-way interactions
- **Temporal features**: Year × Position interactions
- **Missing patterns**: Which tests are skipped together
- **Outlier features**: Distance from position norms

## 1. Setup

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Load Data

In [7]:
train = pd.read_csv('input/train.csv')
test  = pd.read_csv('input/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

Train shape: (2781, 16)
Test shape:  (696, 15)


## 3. Advanced Feature Engineering

In [8]:
perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

def engineer_features(df):
    df = df.copy()
    
    # === V6 Features (baseline) ===
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    for col in perf_cols:
        df[f'{col}_missing'] = df[col].isna().astype(int)
    df['n_missing'] = df[perf_cols].isna().sum(axis=1)
    
    for col in perf_cols:
        grp = df.groupby('Position')[col]
        df[f'{col}_pos_zscore'] = (df[col] - grp.transform('mean')) / (grp.transform('std') + 1e-6)
    
    df['speed_score'] = (1.0 / (df['Sprint_40yd'] + 1e-6) +
                         1.0 / (df['Shuttle'] + 1e-6) +
                         1.0 / (df['Agility_3cone'] + 1e-6))
    
    age_grp = df.groupby('Position')['Age']
    df['age_pos_zscore'] = (df['Age'] - age_grp.transform('mean')) / (age_grp.transform('std') + 1e-6)
    df['height_x_weight'] = df['Height'] * df['Weight']
    df['sprint_x_vertical'] = df['Sprint_40yd'] * df['Vertical_Jump']
    df['sprint_x_broad'] = df['Sprint_40yd'] * df['Broad_Jump']
    
    # === NEW: Polynomial Features ===
    df['sprint_squared'] = df['Sprint_40yd'] ** 2
    df['vertical_squared'] = df['Vertical_Jump'] ** 2
    df['weight_squared'] = df['Weight'] ** 2
    df['bmi_squared'] = df['BMI'] ** 2
    
    # === NEW: Position-Specific Ratios ===
    df['weight_height_ratio'] = df['Weight'] / (df['Height'] + 1e-6)
    df['bench_weight_ratio'] = df['Bench_Press_Reps'] / (df['Weight'] + 1e-6)
    df['vertical_height_ratio'] = df['Vertical_Jump'] / (df['Height'] + 1e-6)
    df['broad_height_ratio'] = df['Broad_Jump'] / (df['Height'] + 1e-6)
    
    # === NEW: Performance Consistency (std of z-scores) ===
    zscore_cols = [f'{c}_pos_zscore' for c in perf_cols]
    df['zscore_std'] = df[zscore_cols].std(axis=1)
    df['zscore_mean'] = df[zscore_cols].mean(axis=1)
    df['zscore_max'] = df[zscore_cols].max(axis=1)
    df['zscore_min'] = df[zscore_cols].min(axis=1)
    
    # === NEW: Elite Athlete Flags (top 10% and 25% by position) ===
    for col in perf_cols:
        # For sprint/agility/shuttle, lower is better
        if col in ['Sprint_40yd', 'Agility_3cone', 'Shuttle']:
            df[f'{col}_top10'] = (df.groupby('Position')[col].rank(pct=True) <= 0.10).astype(int)
            df[f'{col}_top25'] = (df.groupby('Position')[col].rank(pct=True) <= 0.25).astype(int)
        else:  # Higher is better
            df[f'{col}_top10'] = (df.groupby('Position')[col].rank(pct=True) >= 0.90).astype(int)
            df[f'{col}_top25'] = (df.groupby('Position')[col].rank(pct=True) >= 0.75).astype(int)
    
    df['n_top10'] = sum(df[f'{c}_top10'] for c in perf_cols)
    df['n_top25'] = sum(df[f'{c}_top25'] for c in perf_cols)
    
    # === NEW: Multi-Metric Interactions (3-way) ===
    df['sprint_vertical_broad'] = df['Sprint_40yd'] * df['Vertical_Jump'] * df['Broad_Jump']
    df['height_weight_bmi'] = df['Height'] * df['Weight'] * df['BMI']
    df['speed_power'] = df['speed_score'] * df['Vertical_Jump']
    
    # === NEW: Temporal Features (Year interactions) ===
    df['year_normalized'] = (df['Year'] - df['Year'].min()) / (df['Year'].max() - df['Year'].min() + 1e-6)
    df['year_x_sprint'] = df['year_normalized'] * df['Sprint_40yd']
    df['year_x_vertical'] = df['year_normalized'] * df['Vertical_Jump']
    df['year_x_bmi'] = df['year_normalized'] * df['BMI']
    
    # === NEW: Missing Patterns ===
    df['speed_tests_missing'] = (df['Sprint_40yd'].isna() & df['Shuttle'].isna()).astype(int)
    df['jump_tests_missing'] = (df['Vertical_Jump'].isna() & df['Broad_Jump'].isna()).astype(int)
    df['all_tests_present'] = (df['n_missing'] == 0).astype(int)
    
    # === NEW: Outlier Features (distance from position median) ===
    for col in ['Height', 'Weight', 'BMI']:
        pos_median = df.groupby('Position')[col].transform('median')
        df[f'{col}_dist_from_median'] = np.abs(df[col] - pos_median)
    
    # === NEW: Extreme value flags (>2 std from mean) ===
    for col in perf_cols:
        df[f'{col}_extreme'] = (np.abs(df[f'{col}_pos_zscore']) > 2).astype(int)
    
    return df

train = engineer_features(train)
test  = engineer_features(test)
print('Feature engineering done. Train shape:', train.shape)

Feature engineering done. Train shape: (2781, 80)


## 4. Preprocessing

In [9]:
# Encode all categorical columns
cat_cols = ['Player_Type', 'Position_Type', 'Position', 'School']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# Exclude Id and Drafted
features = [c for c in train.columns if c not in ['Id', 'Drafted']]

X      = train[features]
y      = train['Drafted']
X_test = test[features]

print(f'Total features: {len(features)}')

Total features: 78


## 5. Hyperparameter Tuning with Optuna (10-fold)

In [10]:
def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbose':           -1,
        'n_jobs':            -1,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            params, dtrain,
            num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        oof[val_idx] = model.predict(X_val)

    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest OOF AUC: {study.best_value:.5f}')
print('Best params:', study.best_params)

Best trial: 26. Best value: 0.831791: 100%|██████████| 50/50 [07:21<00:00,  8.83s/it]


Best OOF AUC: 0.83179
Best params: {'learning_rate': 0.05733919457526295, 'num_leaves': 104, 'min_child_samples': 43, 'feature_fraction': 0.6344363768868796, 'bagging_fraction': 0.6709624215348489, 'bagging_freq': 2, 'lambda_l1': 3.242374315807894e-05, 'lambda_l2': 2.3840116472146857e-05}


## 6. Train Final Model (10-fold)

In [11]:
best_params = {
    'objective': 'binary',
    'metric':    'auc',
    'verbose':   -1,
    'n_jobs':    -1,
    **study.best_params
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        best_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(X_test) / 10

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold+1:2d}  AUC: {fold_auc:.5f}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOverall OOF AUC: {oof_auc:.5f}')
print(f'Improvement over V6: {oof_auc - 0.83096:+.5f}')

Fold  1  AUC: 0.81052
Fold  2  AUC: 0.83625
Fold  3  AUC: 0.85009
Fold  4  AUC: 0.88724
Fold  5  AUC: 0.85782
Fold  6  AUC: 0.84796
Fold  7  AUC: 0.76888
Fold  8  AUC: 0.81582
Fold  9  AUC: 0.82024
Fold 10  AUC: 0.87948

Overall OOF AUC: 0.83179
Improvement over V6: +0.00083


## 7. Save V7 Submission

In [12]:
submission = pd.read_csv('input/sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv('submission_V7_advanced.csv', index=False)
print('submission_V7_advanced.csv saved!')
submission.head()

submission_V7_advanced.csv saved!


,Id,Drafted
0,2781,0.720793
1,2782,0.842419
2,2783,0.855732
3,2784,0.895982
4,2785,0.754781
